# einops-reduce — ex8: argmax-via-reduce (no torch.argmax) with intermediate prints

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. Running the final beacon cell reports progress against the `Einops: Reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, op)` collapses one or more named axes with a reduction `op` ∈ `{'mean', 'sum', 'max', 'min', 'prod'}`. Drop an axis name on the right side to reduce it; keep it inside parentheses on the left and decompose first to do windowed pooling.

The exercises below stop being about *which op?* and start being about *reduce as part of a larger pipeline* — pyramid pooling, per-channel normalization, argmax-without-`torch.argmax`, top-k by repeated masked max. Each one needs visualization or print-debug to be solvable in your head.

### Exercise 8 — argmax-via-reduce (no torch.argmax) with intermediate prints

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Recover argmax indices using only einops.reduce + arithmetic + boolean masking, printing the intermediate max-mask and index-weighted tensor to see the algorithm work.
> Keywords: argmax, boolean-mask, reduce-max, debug-print
> ```

**KCs targeted:** `reduce-max`, `reduce-keepdim-with-parens`

`torch.argmax` is a black box — it returns indices, you have no idea what happened. Reconstructing argmax from `reduce + max + broadcast + multiply` makes the algorithm transparent and doubles as a great `reduce`-keepdim exercise.

Implement `ex8_argmax_via_reduce(x)` for a 2-D tensor `x` of shape `(R, C)`, returning a 1-D tensor of column indices (one per row), **without calling `torch.argmax`** or `.argmax`. The recipe:

1. `max_per_row = reduce(x, 'r c -> r ()', 'max')` — shape `(R, 1)`, broadcasts back over `x`.
2. `is_max = (x == max_per_row)` — boolean mask, `True` where the value equals its row's max.
3. `col_idx = torch.arange(C)` broadcast over rows.
4. Multiply `is_max.float() * col_idx` then `reduce` with `'max'` over the column axis to pull out the largest True-position. (Why `max` and not `sum`? Ties — if two cells tie for max, you want a single index. Take the rightmost via `max`. If you'd rather match `torch.argmax`'s leftmost-tie rule, use a trick with a tiny epsilon weighted by `-col_idx`.)

Print `max_per_row`, `is_max`, and the masked-index tensor before the final reduce so you can *see* the algorithm. Return the `(R,)` long-tensor of indices.

In [ ]:
def ex8_argmax_via_reduce(x: Tensor) -> Tensor:
    R, C = x.shape
    max_per_row = reduce(x, 'r c -> r ()', 'max')
    is_max = (x == max_per_row)
    col_idx = t.arange(C, device=x.device).expand(R, C)
    # Mask out non-max positions with -1 so reduce-max ignores them
    # (works because col indices are >= 0). For tied maxes, picking
    # 'max' returns the rightmost index.
    masked = t.where(is_max, col_idx, t.full_like(col_idx, -1))
    print('max_per_row:\n', max_per_row)
    print('is_max:\n', is_max)
    print('masked col indices:\n', masked)
    idx = reduce(masked, 'r c -> r', 'max')
    return idx.long()


<details><summary>Solution</summary>

```python
def ex8_argmax_via_reduce(x: Tensor) -> Tensor:
    R, C = x.shape
    max_per_row = reduce(x, 'r c -> r ()', 'max')
    is_max = (x == max_per_row)
    col_idx = t.arange(C, device=x.device).expand(R, C)
    # Mask out non-max positions with -1 so reduce-max ignores them
    # (works because col indices are >= 0). For tied maxes, picking
    # 'max' returns the rightmost index.
    masked = t.where(is_max, col_idx, t.full_like(col_idx, -1))
    print('max_per_row:\n', max_per_row)
    print('is_max:\n', is_max)
    print('masked col indices:\n', masked)
    idx = reduce(masked, 'r c -> r', 'max')
    return idx.long()
```

**Why the `-1` masking trick?** A boolean mask multiplied by indices works *unless* the argmax sits at column 0 — then both the true max-position and all the masked-out positions read as 0, and `reduce-max` can't tell them apart. Using `-1` for masked-out positions makes the true-positive a strict winner.

**Why this is in the curriculum.** Every `argmax`/`top-k` trick in attention code (relative position bias, ALiBi, sparse routing) is a variation on this recipe. Building it once by hand removes the magic.

**Tie-breaking note.** `torch.argmax` returns the *leftmost* tie. Our recipe returns the *rightmost* because `reduce-max` over indices picks the largest. If you need leftmost-tie semantics, subtract `col_idx * eps` from `x` before the whole pipeline.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()